# 01 / Max-Cut — beginner challenge

**Question:** Can a quantum algorithm find a good way to divide a network into two groups?

In this exercise you will **understand Max-Cut**, **turn it into QUBO**, **map it to a quantum cost operator**, **solve it classically**, **run QAOA in a simulator**, and optionally **test the same circuit on IQM hardware**.

**Before you start:** No quantum physics or graph theory is required. Run the cells from top to bottom. Hardware execution is optional and disabled by default. This is a custom QSIP teaching example, not an official IQM tutorial.

## 1 / The idea — what is Max-Cut?

Imagine six people connected by lines indicating who knows whom. You must split them into two groups so that **as many connections as possible run between groups**.

- A **node** (vertex) is one person or object
- An **edge** is a connection between two nodes
- A **graph** is the collection of nodes and edges
- A **cut** is a division of nodes into two groups
- A **crossing edge** connects nodes in different groups and counts toward the score

This example has **6 nodes and 11 edges**. You will draw the network, try an assignment, then calculate its exact optimum.

## 2 / Optimization vocabulary

| Term | Meaning in this exercise |
|---|---|
| **Decision variable** | A choice we can change: which group each node belongs to |
| **Binary variable** `x_i` | 0 for group A, 1 for group B |
| **Objective function** | The number (or total weight) of crossing edges; **maximize** it |
| **Constraint** | Every node must belong to exactly one of the two groups |
| **Feasible solution** | Any valid assignment of all six nodes |
| **Global optimum** | The largest possible cut score over all assignments |
| **Bitstring** | A string of zeros and ones representing one proposed partition |
| **Benchmark** | A classical result used to evaluate the quantum algorithm |

**Check your understanding:** If two connected nodes are assigned `0` and `1`, does their edge count toward the cut? **Yes.** If both are `0`, it does not.

## 3 / From a picture to a mathematical problem

For every node `i`, choose a binary variable $x_i\in\{0,1\}$. Consider one edge connecting nodes `i` and `j`:

| `x_i` | `x_j` | Different groups? | Edge contributes |
|:---:|:---:|:---:|:---:|
| 0 | 0 | No | 0 |
| 0 | 1 | Yes | 1 |
| 1 | 0 | Yes | 1 |
| 1 | 1 | No | 0 |

The expression $x_i+x_j-2x_ix_j$ is **1 only when the groups differ**. Summing over all edges gives:

$$
\begin{aligned}
\text{Maximize}\quad & C(x) = \sum_{(i,j)\in E} w_{ij}\left(x_i+x_j-2x_ix_j\right) \\
\text{subject to}\quad & x\in\{0,1\}^{|V|}.
\end{aligned}
$$

$V$ denotes the set of nodes, $E$ the set of edges, and $x=(x_1,\ldots,x_{|V|})$ the assignment of every node to group 0 or 1. The edge weight $w_{ij}$ is 1 for every edge in this example. This is a **QUBO** (Quadratic Unconstrained Binary Optimization) objective because it is quadratic in binary variables. For six nodes there are $2^6=64$ candidate assignments; exchanging every 0 with 1 leaves the cut unchanged.

## 4 / From Max-Cut to QUBO

**Max-Cut** is the problem (find the best partition). **QUBO** is a way of writing the objective so that numerical and quantum optimization methods can use it.

QUBO stands for **Quadratic Unconstrained Binary Optimization**:

- **Quadratic:** the objective includes terms such as $x_ix_j$, involving at most two binary variables.
- **Unconstrained:** the optimization is over binary assignments without extra equality or inequality constraints. The choice $x_i\in\{0,1\}$ is part of the variable domain.
- **Binary:** every variable is either 0 or 1.

The conventional QUBO form is a **minimization**:

$$
\min_{x\in\{0,1\}^n} x^{\mathsf T}Qx.
$$

Here $Q$ is a matrix of coefficients, and $x$ is the vector of binary decisions. Because $x_i^2=x_i$ for binary variables, diagonal terms in $Q$ represent linear coefficients. We use a **symmetric** $Q$ in the example below, so a coefficient $Q_{ij}$ for $i\ne j$ contributes $2Q_{ij}x_ix_j$.

Max-Cut instead **maximizes** $C(x)$. To use the usual QUBO convention, simply minimize its negative:

$$
\underset{x}{\operatorname{argmax}}\; C(x)
\;=\;
\underset{x}{\operatorname{argmin}}\;[-C(x)].
$$

Both formulations produce exactly the same optimal partition.

### Worked example — three nodes

Take a chain with two edges: an edge of **weight 2** between nodes 0 and 1, and an edge of **weight 1** between nodes 1 and 2.

$$
\begin{aligned}
C(x)&=2(x_0+x_1-2x_0x_1)+(x_1+x_2-2x_1x_2)\\
&=2x_0+3x_1+x_2-4x_0x_1-2x_1x_2.
\end{aligned}
$$

For $x=(0,1,0)$, **both edges cross** and the score is $C(x)=2+1=3$. Its QUBO value is $-3$.

One symmetric QUBO matrix for this example is:

$$
Q=\begin{pmatrix}
-2 & 2 & 0\\
2 & -3 & 1\\
0 & 1 & -1
\end{pmatrix},
\qquad x^{\mathsf T}Qx=-C(x).
$$

**Try it:** Run the next cell. It draws the example, evaluates all eight assignments, and checks that the Max-Cut maximum and QUBO minimum agree.

In [ ]:
# Worked example: three-node path, with two weighted edges
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

example_edges = [(0, 1, 2.0), (1, 2, 1.0)]
Q_example = np.array([[-2., 2., 0.], [2., -3., 1.], [0., 1., -1.]])

def example_cut(bits):
    return sum(w for i, j, w in example_edges if bits[i] != bits[j])

rows = []
for bits in product((0, 1), repeat=3):
    x = np.asarray(bits, dtype=float)
    rows.append((''.join(map(str, bits)), example_cut(bits), float(x @ Q_example @ x)))
    assert np.isclose(rows[-1][1], -rows[-1][2])

print('Assignment | Max-Cut score | QUBO value (-score)')
for label, cut, qubo_value in rows:
    print(f'  {label}      | {cut:4.0f}          | {qubo_value:5.0f}')
print('Best cut =', max(r[1] for r in rows), ' / lowest QUBO =', min(r[2] for r in rows))

fig, ax = plt.subplots(figsize=(7, 2.4))
positions = {0: (0, 0), 1: (1, 0), 2: (2, 0)}
for i, j, w in example_edges:
    ax.plot([positions[i][0], positions[j][0]], [0, 0], color='#D4212E', lw=2.3 + w, zorder=1)
    ax.text((positions[i][0] + positions[j][0]) / 2, 0.14,
            f'weight {w:g}', ha='center', fontsize=10)
for node, bit in enumerate((0, 1, 0)):
    ax.scatter(node, 0, s=1450, c=('#D4212E' if bit == 0 else '#161616'), zorder=3)
    ax.text(node, 0, f'{node}\n({bit})', color='white', ha='center', va='center',
            fontsize=10, fontweight='bold', zorder=4)
ax.set_xlim(-.4, 2.4)
ax.set_ylim(-.35, .45)
ax.set_title('A maximum cut for the weighted three-node chain', loc='left')
ax.axis('off')
plt.show()

## 5 / The network and a trial solution

Run the next two cells. The first creates the synthetic graph; the second assigns each node to one of two groups.

**Read the output:** node labels are identifiers, node colors are group assignments, and an edge contributes to the cut only if its endpoints have different colors. The printed cut score is your objective value, **not** a probability or a quantum result.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'qsip_quantum.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import numpy as np
from qsip_quantum import (
    starter_graph, weighted_edges, draw_graph, cut_score, brute_force,
    qaoa_circuit, simulate_counts, best_from_counts, optimize_p1,
    qaoa_expected_score, bits_from_qiskit
)
N_NODES = 6  # Change to 4 if hardware transpilation becomes too expensive
G = starter_graph().subgraph(range(N_NODES)).copy()
edges = weighted_edges(G)
n = G.number_of_nodes()
print(f'{n} nodes, {len(edges)} edges')

In [ ]:
# A trial partition — change the zeros and ones and rerun
trial_bits = (0, 1, 0, 0, 0, 1)[:n]  # six-node cut score = 6; can you improve it?
trial_score = cut_score(trial_bits, edges)
print('Your partition:', trial_bits)
print(f'Crossing edges: {int(trial_score)} of {len(edges)}')
print('Try changing the assignment, then compare it with the exact optimum below.')
fig, ax = draw_graph(G, trial_bits, title='Trial partition / red and black nodes are two groups')
plt.show()

## 6 / Find the exact answer classically

Because there are only six binary decisions, the computer can check all $2^6=64$ assignments. The result is the **largest possible number of crossing edges**.

Compare the next plot with your trial partition. The maximum is **8** for this fixed graph. The exact result is the baseline for judging the quantum approach; larger problems often require more sophisticated classical methods.

In [ ]:
(exact_score, exact_bits), exhaustive_results = brute_force(
    n, lambda bits: cut_score(bits, edges)
)
if N_NODES == 6:
    assert exact_score == 8.0
print('Exact Max-Cut score:', exact_score)
print('One optimal assignment:', exact_bits)
fig, ax = draw_graph(G, exact_bits, title='An exact optimal partition')
plt.show()
print('Meaning: no partition of this graph has a cut score above', int(exact_score))

## 7 / From QUBO to an Ising Hamiltonian

A **Hamiltonian** is an operator that assigns an energy (or, more generally here, an objective value) to quantum states. To make a binary optimization problem usable by QAOA, map every binary decision to a qubit:

$$
x_i\quad\longrightarrow\quad\frac{I-Z_i}{2}.
$$

- $I$ is the identity operator (leaves a state unchanged).
- $Z_i$ is the **Pauli-Z** operator on qubit $i$: it has eigenvalue $+1$ on $|0\rangle$ and $-1$ on $|1\rangle$.
- Thus the expression $(I-Z_i)/2$ gives 0 for $|0\rangle$ and 1 for $|1\rangle$.

For Max-Cut, the **cost Hamiltonian** that corresponds to the maximized cut value is:

$$
H_C=\sum_{(i,j)\in E}\frac{w_{ij}}{2}(I-Z_i Z_j).
$$

For a pair of qubits, $Z_iZ_j$ is $+1$ when their measured bits agree and $-1$ when they differ. So each term of $H_C$ contributes **zero for same-group nodes** and **$w_{ij}$ for different-group nodes**—just like Max-Cut.

**Sign convention:** The QUBO in Section 4 is $-C(x)$ because QUBO is conventionally minimized. In the code, QAOA instead uses the **maximization cost** $+C(x)$, represented by $H_C$. These are equivalent optimization targets, provided the circuit sign and classical objective are consistent.

### Follow the same problem through four stages

| Stage | Meaning | In this notebook |
|---|---|---|
| **Max-Cut** | Choose the best partition | Maximize $C(x)$ |
| **QUBO** | Express it using binary quadratic terms | Minimize $-C(x)=x^{\mathsf T}Qx$ |
| **Ising / Hamiltonian** | Encode the objective with qubit operators | $H_C=\sum w_{ij}(I-Z_iZ_j)/2$ |
| **QAOA** | Use a tunable quantum circuit to seek high-scoring assignments | Cost layer, mixer layer, measurement and classical optimization |

The worked three-node case and the larger six-node challenge follow the same procedure.

### Check your understanding — what changes from one stage to the next?

Consider one unweighted edge connecting nodes 0 and 1.

**Problem (Max-Cut):** award 1 point if the nodes belong to different groups.

**QUBO (binary):** with $x_0,x_1\in\{0,1\}$, the score is $C=x_0+x_1-2x_0x_1$. For conventional QUBO minimization, use $-C$.

**Ising (spin variables):** define $s_i=1-2x_i\in\{-1,+1\}$. The very same edge has score

$$
C=\frac{1-s_0s_1}{2}.
$$

The Ising model is **not a second algorithm**. It is the same score written in $-1/+1$ variables. On a quantum computer, replace $s_i$ by the Pauli-$Z_i$ operator to get $H_C=(I-Z_0Z_1)/2$.

**QAOA (algorithm):** use this cost operator inside a *tunable* quantum circuit. “Tunable” means its rotation angles can be changed between runs by a classical optimizer.

Try the four possibilities mentally: `00` and `11` score 0; `01` and `10` score 1. **All three formulations give the same score.**


## 8 / How QAOA searches for good partitions

**QAOA** (Quantum Approximate Optimization Algorithm) is a **hybrid quantum–classical** method: the quantum circuit generates a distribution over candidate partitions, while a classical optimizer **changes the circuit’s adjustable rotation angles** to improve the expected score. The circuit is *tunable* because these angles are not fixed.

1. **Initialize:** apply Hadamard gates so all bitstrings start with equal probability amplitudes.
2. **Cost layer:** apply $U_C(\gamma)=e^{-i\gamma H_C}$; different objective values acquire different quantum phases.
3. **Mixer layer:** apply $U_M(\beta)=e^{-i\beta\sum_i X_i}$; $X_i$ is the Pauli-X operator, which mixes computational-basis states.
4. **Measure:** a circuit execution (**shot**) returns one classical bitstring, such as `010101`.
5. **Update:** calculate the average cut score and let a classical optimizer change $\gamma$ and $\beta$. Repeat. In this notebook, the optimizer evaluates an **ideal statevector simulator**, not repeated hardware jobs.

An individual **shot** is one measurement returning one candidate bitstring; multiple shots show how often each answer is obtained. The **best sampled score** is the best answer observed; the **mean score** tells us how consistently the algorithm finds good answers.

The cost and mixer layers may be repeated **$p$ times**. Our starter circuit uses $p=1$ (one cost layer and one mixer layer). Unlike the logical circuit *depth*, $p$ is the number of QAOA layer pairs.

The circuit uses quantum **interference** to change sampling probabilities; it does **not** read out all solutions at once. **A better distribution is not guaranteed**, and a classical method can perform better. That is why we compare with the exact classical optimum.

### Visual summary

In [ ]:
# Conceptual flow of one QAOA optimization iteration
from matplotlib.patches import FancyBboxPatch
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 2.0))
items = [
    ('Problem', 'Max-Cut'),
    ('Encoding', 'Cost operator'),
    ('Quantum circuit', 'Cost + mixer'),
    ('Readout', 'Sample bitstrings'),
    ('Classical step', 'Update angles'),
]
xs = [.02, .225, .43, .635, .84]
for k, ((heading, detail), x) in enumerate(zip(items, xs)):
    rect = FancyBboxPatch((x, .30), .15, .42, boxstyle='round,pad=0.015,rounding_size=0.018',
                          linewidth=1.1, edgecolor='#202020', facecolor='white')
    ax.add_patch(rect)
    ax.text(x+.075, .57, heading, fontsize=10, fontweight='bold', ha='center')
    ax.text(x+.075, .43, detail, fontsize=9, ha='center')
    if k < len(items)-1:
        ax.annotate('', xy=(xs[k+1]-.009, .51), xytext=(x+.17, .51),
                    arrowprops=dict(arrowstyle='->', color='#D4212E', lw=1.4))
ax.text(.51, .08, 'Repeat: the classical optimizer adjusts γ and β to improve the expected score',
        ha='center', color='#D4212E', fontsize=10)
ax.set(xlim=(0, 1.02), ylim=(0, 1))
ax.axis('off')
plt.show()

## 9 / Build the QAOA circuit

For each edge, the circuit uses the Max-Cut cost term $w_{ij}(I-Z_iZ_j)/2$ from Section 7. The **cost unitary** is $U_C(\gamma)=e^{-i\gamma H_C}$. The **mixer unitary** is $U_M(\beta)=e^{-i\beta\sum_i X_i}$.

Our helper implements these layers using Qiskit rotations (`rzz` for graph edges and `rx` for qubits). Its `gamma` and `beta` parameters control the quantum interference pattern. This version optimizes the **expected cut value** and samples the resulting bitstrings; it does not use IQM's separate high-level QAOA library.

Try changing `gamma` and `beta` in the next cell and see how the results change.

In [ ]:
gamma, beta = 0.75, 0.45  # TODO: experiment with different values
circuit = qaoa_circuit(n, edges, gammas=(gamma,), betas=(beta,))
print('Logical circuit depth:', circuit.depth())
print(circuit.draw(output='text', fold=100))

## 10 / Sample the simulated circuit

The simulator runs the QAOA circuit and produces **bitstrings**—one bit for each node's community assignment. A **shot** is one circuit run and measurement.

- The **best observed score** is the largest cut value among the sampled assignments.
- The **counts plot** shows which bitstrings occurred most often; frequency is not the same as solution quality.

This is still a **classical simulation of a quantum circuit**, not a real IQM measurement.

In [ ]:
counts = simulate_counts(circuit, shots=1024)
observed_score, observed_label, observed_count = best_from_counts(
    counts, lambda bits: cut_score(bits, edges)
)
print('Best observed score:', observed_score, '/ exact:', exact_score)
print('Best observed bitstring (Qiskit order):', observed_label)
print('Shots at that bitstring:', observed_count)
assert observed_score <= exact_score + 1e-10

In [ ]:
labels = sorted(counts, key=counts.get, reverse=True)[:12]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(labels, [counts[label] for label in labels], color='#151515')
ax.set(xlabel='Measured bitstring', ylabel='Counts', title='Sampled QAOA solutions')
plt.xticks(rotation=40)
plt.tight_layout()

## 11 / Optimize the QAOA angles

The circuit is **tunable**: the numerical angles $\gamma$ (cost layer) and $\beta$ (mixing layer) can change without changing the graph or the goal. A **classical** optimization routine (Nelder–Mead) tries different angle pairs and evaluates them on an **ideal quantum simulator** to increase the **expected cut value**. The circuit does not autonomously learn; Python updates its parameters between evaluations.

The next plot compares:

- **Initial QAOA mean:** average score of the first simulated circuit
- **Optimized QAOA mean:** average score after optimizing $\gamma$ and $\beta$
- **Classical optimum:** guaranteed maximum score of 8

An individual sampled solution may reach 8 even if the **average** QAOA score is lower. That does not establish a quantum advantage.

In [ ]:
result = optimize_p1(n, edges, maxiter=75)
opt_gamma, opt_beta = result.x
expected = qaoa_expected_score(n, edges, opt_gamma, opt_beta)
print('Optimized parameters:', np.round(result.x, 3))
print('Simulated expected cut:', round(expected, 3))
opt_circuit = qaoa_circuit(n, edges, (opt_gamma,), (opt_beta,))
opt_counts = simulate_counts(opt_circuit, shots=1024, seed=2026)
best_score, best_label, _ = best_from_counts(
    opt_counts, lambda bits: cut_score(bits, edges)
)
print('Best measured cut:', best_score)
print('Approximation ratio (best sample / exact):', round(best_score / exact_score, 3))
fig, ax = draw_graph(G, bits_from_qiskit(best_label), title='Best sampled QAOA partition')

In [ ]:
# Compare the exact optimum with the *mean* QAOA sample score
score_fn = lambda bits: cut_score(bits, edges)
def sampled_mean(samples):
    return sum(c * score_fn(bits_from_qiskit(label)) for label, c in samples.items()) / sum(samples.values())
initial_mean = sampled_mean(counts)
optimized_mean = sampled_mean(opt_counts)
fig, ax = plt.subplots(figsize=(7.8, 3))
ax.bar(['Initial QAOA', 'Optimized QAOA', 'Exact optimum'], [initial_mean, optimized_mean, exact_score], color=['#A8A8A8','#D4212E','#181818'])
ax.set_ylabel('Cut score'); ax.set_title('Like-for-like comparison / same Max-Cut objective')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
print(f'Initial mean: {initial_mean:.3f}; optimized mean: {optimized_mean:.3f}; exact optimum: {exact_score:.0f}')

## 12 / Your challenge

Try changing the QAOA angles or one edge of the graph. Then report the **exact cut score**, the **mean simulated QAOA cut score**, the **best sampled score**, and whether optimization improved the average.

If time allows, change the number of QAOA layers or test the same optimized circuit on IQM hardware locally.

---
## Optional / Real IQM hardware (local VS Code only)

Everything above runs without a token in Binder. If you have IQM Resonance access, you can **open this same notebook locally in VS Code** and execute the optimized circuit on a real quantum processor.

1. Install the dependencies from `requirements-hardware.txt` in a local Python 3.11 or 3.12 environment.
2. Run the simulator cells above.
3. Set `RUN_ON_IQM = True` and `IQM_QUANTUM_COMPUTER = 'garnet'` (or another processor assigned to you).
4. Enter your token at the hidden prompt. Inspect the transpiled circuit before enabling `CONFIRM_QPU_SUBMISSION = True`.

**Never enter an IQM token in public Binder.** QPU execution may consume credits or incur charges.

In [ ]:
# Stage B / explicit hardware opt-in. No token is used while this is False.
import os
RUN_ON_IQM = False
CONFIRM_QPU_SUBMISSION = False
QPU_SHOTS = 256
IQM_SERVER_URL = os.getenv('IQM_SERVER_URL', 'https://resonance.iqm.tech')
IQM_QUANTUM_COMPUTER = os.getenv('IQM_QUANTUM_COMPUTER', '')

print('IQM hardware stage:', 'ENABLED' if RUN_ON_IQM else 'OFF — simulation complete')


In [ ]:
# Stage C / hidden personal token + backend discovery. Run in trusted Jupyter only.
# Do not enable RUN_ON_IQM in a public mybinder.org session.
from getpass import getpass
from qiskit import transpile

qpu_backend = None
qpu_circuit = None
if RUN_ON_IQM:
    # This is a precaution; always check the real hosting environment yourself.
    if 'mybinder' in str(os.getenv('JUPYTERHUB_HOST', '')).lower() or os.getenv('BINDER_LAUNCH_HOST'):
        raise RuntimeError('Do not provide personal IQM credentials to public Binder. Use local VS Code or trusted Jupyter.')
    try:
        from iqm.qiskit_iqm import IQMProvider
    except ImportError as exc:
        raise ImportError('Install the IQM adapter in this trusted environment: pip install -r requirements-hardware.txt') from exc
    qpu_name = IQM_QUANTUM_COMPUTER.strip() or input('IQM processor name (e.g., garnet): ').strip()
    if not qpu_name:
        raise ValueError('Processor name required. Check IQM Resonance or ask the facilitators.')
    personal_token = os.environ.get('IQM_TOKEN') or getpass('Your IQM API token (input hidden): ')
    if not personal_token:
        raise ValueError('Your IQM token is required for hardware access.')
    try:
        provider = IQMProvider(IQM_SERVER_URL, quantum_computer=qpu_name, token=personal_token)
        qpu_backend = provider.get_backend()
    finally:
        del personal_token  # Never print credentials or save them to the notebook.
    qpu_circuit = transpile(opt_circuit, backend=qpu_backend, optimization_level=1)
    print('Connected to:', qpu_name, '| physical qubits:', qpu_backend.num_qubits)
    print('Logical depth:', opt_circuit.depth(), '| transpiled depth:', qpu_circuit.depth())
    print('Mapped operations:', dict(qpu_circuit.count_ops()))
else:
    print('Skipped — no IQM token requested. You can stop after simulation.')


In [ ]:
# Stage D / submit ONLY after manually setting both switches to True.
qpu_counts = None
if RUN_ON_IQM and CONFIRM_QPU_SUBMISSION and qpu_circuit is not None:
    job = qpu_backend.run(qpu_circuit, shots=QPU_SHOTS)
    print('Submitted IQM job:', job.job_id())
    qpu_counts = job.result().get_counts()
    print('Hardware shots collected:', sum(qpu_counts.values()))
else:
    print('No hardware job submitted. Set CONFIRM_QPU_SUBMISSION=True if you are ready.')


In [ ]:
# Stage E / compare classical optimum, simulator and the QPU on the SAME objective.
score_fn = lambda bits: cut_score(bits, edges)
simulated_counts = opt_counts

def average_objective(samples):
    total = sum(samples.values())
    return sum(score_fn(bits_from_qiskit(bitstring)) * frequency
               for bitstring, frequency in samples.items()) / total

print('Objective: Max-Cut value')
print('Classical optimum:', round(exact_score, 6))
print('Simulated mean:', round(average_objective(simulated_counts), 6))
if qpu_counts:
    print('Real IQM mean:', round(average_objective(qpu_counts), 6))
    value, best_bitstring, _ = best_from_counts(qpu_counts, score_fn)
    print('Best observed real-QPU value:', round(value, 6), 'bitstring:', best_bitstring)
else:
    print('Real IQM: not run — simulation-only experiment completed.')


### If the hardware stage is skipped

That is fine. With `RUN_ON_IQM = False`, you can run all cells without a token. If you run locally, set `IQM_QUANTUM_COMPUTER = 'garnet'` (or another processor assigned to you), enable `RUN_ON_IQM`, and only submit after confirming access and charges.

Do not save, share or commit a notebook containing credentials or private job metadata.

## References and provenance

**Custom QSIP teaching notebook**, not a reproduction of IQM code and not a scientifically validated demonstration of quantum advantage.

1. Farhi, E., Goldstone, J. & Gutmann, S. (2014). *A Quantum Approximate Optimization Algorithm*. [arXiv:1411.4028](https://arxiv.org/abs/1411.4028) — QAOA framework.
2. Lucas, A. (2014). *Ising formulations of many NP problems*. [Frontiers in Physics](https://doi.org/10.3389/fphy.2014.00005) — binary/Ising formulations of combinatorial optimization.
3. [IQM Quantum Computers — Sparse Maxcut tutorial](https://docs.iqm.tech/4.6/iqm-qaoa/Sparse%20Maxcut.html) — related official example; the QSIP notebook instead uses the custom `qsip_quantum.py` helper.
4. [IQM — Qiskit client and hardware user guide](https://docs.iqm.tech/iqm-client/user_guide_qiskit.html) — optional backend execution.
5. [Qiskit Aer documentation](https://qiskit.github.io/qiskit-aer/) — local circuit simulation.

**Interpretation:** Report comparisons and limitations, not an unsupported claim of quantum speedup.

## Optional extension — training QAOA on real IQM hardware

**What this notebook does:** A classical optimizer tunes $\gamma$ and $\beta$ using an *ideal quantum simulator*. The optional IQM cells then execute the **already-optimized circuit** on a real QPU to compare its noisy measurements with simulation.

**What it does not do:** It does **not** rerun the parameter-optimization loop on the physical QPU. That more advanced approach submits many different circuits and uses the measured objective value to update the next set of angles; it can consume substantial QPU time and credits.

Official examples for interested participants:

- [IQM — Training the QAOA](https://docs.iqm.tech/4.6/iqm-qaoa/Training%20the%20QAOA.html)
- [IQM — Sparse Max-Cut](https://docs.iqm.tech/4.6/iqm-qaoa/Sparse%20Maxcut.html)

**Do not put personal IQM tokens into public Binder.** Run hardware experiments from a trusted local Jupyter session only.
